# Feature Type Definitions

**`FeatureTypeDefinition`** (plugin_id `collection:feature_type`) is a driver-agnostic schema
decorator that declares which fields are exposed, filtered, or enriched at each
entity level: `catalog`, `collection`, `item`, or `asset`.

It acts as a two-way contract:
- **At storage time** — drivers use it to create backing storage with the right structure
- **At read time** — extensions use it to shape OGC / STAC responses

## Key concepts

| Concept | Purpose |
|---|---|
| `FieldDefinition` | Describes one queryable field: name, data_type, capabilities |
| `FieldCapability` | `filterable`, `sortable`, `groupable`, `aggregatable`, `spatial`, `indexed` |
| `EntityLevel` | `catalog`, `collection`, `item`, `asset` |
| `exclude_fields` | Field names to strip from the driver's native schema |
| `metadata_fields` | Extra metadata to attach (keywords, summaries, etc.) |

## Driver introspection: `get_entity_fields()`

All four drivers (PG, Elasticsearch, Iceberg, DuckDB) implement
`get_entity_fields(catalog_id, collection_id, entity_level="item")`
which returns `Dict[str, FieldDefinition]` — the live schema of the backing store.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-feature-type"
COLLECTION_ID = "climate-stations"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Feature Type Demo"})
print("Catalog:", r.status_code)

---
## 1. Declare a feature type at collection creation

Pass `feature_type` in the collection body.  It is persisted via the config
waterfall and passed to the driver's `ensure_storage()` to pre-create the schema.

In [ ]:
feature_type = {
    "level": "item",
    "fields": {
        "station_id": {
            "name": "station_id",
            "data_type": "string",
            "capabilities": ["filterable", "sortable", "indexed"],
        },
        "temperature": {
            "name": "temperature",
            "data_type": "numeric",
            "capabilities": ["filterable", "sortable", "aggregatable"],
            "aggregations": ["min", "max", "avg"],
        },
        "humidity": {
            "name": "humidity",
            "data_type": "numeric",
            "capabilities": ["filterable", "aggregatable"],
        },
        "geom": {
            "name": "geom",
            "data_type": "geometry",
            "capabilities": ["spatial"],
        },
        "_internal_token": {
            "name": "_internal_token",
            "data_type": "string",
            "expose": False,          # hidden from public OGC/STAC responses
            "capabilities": [],
        },
    },
    "exclude_fields": ["raw_payload"],   # strip from driver schema
    "metadata_fields": {
        "keywords": ["climate", "temperature", "humidity", "WMO"],
        "summaries": {"temperature": {"minimum": -50, "maximum": 60}},
    },
}

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Climate Stations",
    "license": "CC-BY-4.0",
    "feature_type": feature_type,   # embedded at creation
})
print("Collection:", r.status_code)

---
## 2. Update feature type via config API

The feature type can be updated independently of the collection metadata.

In [ ]:
# Add a new field after collection already exists
feature_type["fields"]["wind_speed"] = {
    "name": "wind_speed",
    "data_type": "numeric",
    "capabilities": ["filterable", "aggregatable"],
    "aggregations": ["avg", "max"],
}

r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/collection:feature_type",
    json=feature_type,
)
print("Feature type updated:", r.status_code)

---
## 3. Introspect the live driver schema

`GET /storage/catalogs/{cat}/collections/{coll}/fields?entity_level=item`
calls the driver's `get_entity_fields()` and returns the live backing schema as
`FieldDefinition` objects — reflecting the actual columns or index mappings.

In [ ]:
r = await client.get(
    f"/storage/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/fields",
    params={"entity_level": "item"},
)
print("Status:", r.status_code)
if r.status_code == 200:
    fields = r.json()
    print(f"\nDriver reports {len(fields)} fields:")
    for name, fd in fields.items():
        caps = ", ".join(fd.get("capabilities", []))
        print(f"  {name:30s} type={fd.get('data_type'):10s} caps=[{caps}]")

---
## 4. Queryables — OGC API Features integration

The OGC `/queryables` endpoint uses `introspect_schema()` (which returns
`FieldDefinition` objects) to build the JSON Schema response.  Only fields
with `expose=True` (the default) appear in the public response.

In [ ]:
# Ingest a few items so the schema is populated
import random
features = [
    {
        "type": "Feature",
        "id": f"WMO-{i:05d}",
        "geometry": {"type": "Point", "coordinates": [10 + random.uniform(-20, 20), 45 + random.uniform(-30, 30)]},
        "properties": {
            "station_id": f"WMO-{i:05d}",
            "temperature": round(random.uniform(-10, 40), 1),
            "humidity": round(random.uniform(20, 95), 1),
            "wind_speed": round(random.uniform(0, 30), 1),
        },
    }
    for i in range(10)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
print("Items written:", r.status_code)

# Queryables — shows only exposed fields
r = await client.get(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables")
print("Queryables status:", r.status_code)
if r.status_code == 200:
    q = r.json()
    print(f"Exposed queryable fields: {list(q.get('properties', {}).keys())}")
    # _internal_token should NOT appear (expose=False)

---
## 5. Entity levels: `catalog` and `asset`

Feature type definitions work at all entity levels.  At `catalog` level they
declare catalog-wide metadata fields.  At `asset` level they define which
asset metadata fields are exposed.

In [ ]:
# Declare asset-level feature type for this collection
asset_feature_type = {
    "level": "asset",
    "fields": {
        "href": {"name": "href", "data_type": "string", "capabilities": ["filterable"]},
        "media_type": {"name": "media_type", "data_type": "string", "capabilities": ["filterable", "groupable"]},
        "file_size": {"name": "file_size", "data_type": "integer", "capabilities": ["filterable", "sortable", "aggregatable"]},
        "checksum": {"name": "checksum", "data_type": "string", "capabilities": [], "expose": False},
    },
}

r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/asset:feature_type",
    json=asset_feature_type,
)
print("Asset feature type stored:", r.status_code)

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print("Cleanup:", r.status_code)
await client.aclose()